In [1]:
# 2026-07-13 sofia

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from dotenv import load_dotenv
load_dotenv()

import voyageai
client = voyageai.Client()
model = "voyage-4-lite"

In [4]:
from anthropic import Anthropic

anthropic_client = Anthropic()
anthropic_model = "claude-haiku-4-5"

In [5]:
from text_chunker import TextChunker

text_chunker = TextChunker()

In [6]:
from chat_helper import ChatHelper

chat_helper = ChatHelper(anthropic_client, anthropic_model)

In [7]:
from context_provider import ContextProvider

context_provider = ContextProvider(chat_helper)

In [8]:
from embedding_generator import EmbeddingGenerator

embedding_generator = EmbeddingGenerator(client, model)

In [9]:
from vector_index import VectorIndex

vector_index = VectorIndex(embedding_fn=embedding_generator.generate_embedding)

In [10]:
from bm25_index import BM25Index

bm25_index = BM25Index()

In [11]:
from retriever import Retriever

retriever = Retriever(bm25_index, vector_index)

In [12]:
from reranker import Reranker

reranker = Reranker(chat_helper)

In [13]:
with open("./report.md", "r") as f:
    text = f.read()

In [14]:
# 1. Chunk the text by section

chunks = text_chunker.chunk_by_section(text)

In [15]:
# 2. Add context to each chunk, then add to the retriever

num_start_chunks = 2
num_prev_chunks = 2

for i, chunk in enumerate(chunks):
    context_parts = []

    # Initial set of chunks from the start of the doc
    context_parts.extend(chunks[: min(num_start_chunks, len(chunks))])

    # Additional chunks ahead of the current chunk we're contextualizing
    start_idx = max(0, i - num_prev_chunks)
    context_parts.extend(chunks[start_idx:i])

    context = "\n".join(context_parts)

    contextualized_chunk = context_provider.add_context(chunk, context)
    # print(contextualized_chunk)
    contextualized_chunk_text = contextualized_chunk.content[0].text
    
    retriever.add_document({"content": contextualized_chunk_text})

In [16]:
# 3. Retrieve results combining scores from Vector Index and BM25 Index

query_text = "What did the engineering team do with INC-2023-Q4-111?"
results = retriever.search(query_text, 3)

for doc, score in results:
    print(score, "\n\n", doc["content"], "\n---\n")

0.03278688524590164 

 This chunk is Section 10 of the Annual Interdisciplinary Research Review, covering Cybersecurity Analysis. It documents a contained targeted intrusion attempt (INC-2023-Q4-011) attributed to the ShadowNet Syndicate, which exploited spear-phishing to target the finance department and attempted lateral movement toward financial servers. The incident illustrates organizational cybersecurity resilience and highlights vulnerabilities in protecting sensitive financial and research data across departments. 
---

0.03149801587301587 

 This chunk is Section 6 of the Annual Interdisciplinary Research Review, detailing the Product Engineering team's completion of specifications for the Model Zircon-5 platform. It represents a major milestone in the organization's product development pipeline, building on advances from Software Engineering (Section 2) and incorporating potential innovations from Scientific Experimentation (Section 4) regarding Material Composite XT-5 for fu